In [26]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, classification_report

In [27]:
np.random.seed(42)

n = 200

df = pd.DataFrame(
    {
        '年龄': np.random.randint(18, 65, n).astype(float),
        '收入': np.random.randint(3000, 50000, n).astype(float),
        '城市': np.random.choice(['北京', '上海', '深圳'], n),
        '是否购买': np.random.choice([0, 1], n),
    }
)

df

,年龄,收入,城市,是否购买
0,56.0,47417.0,上海,0
1,46.0,26938.0,上海,0
2,32.0,26664.0,深圳,1
3,60.0,4636.0,深圳,0
4,25.0,23080.0,深圳,1
...,...,...,...,...
195,49.0,48091.0,深圳,1
196,49.0,7780.0,深圳,0
197,41.0,5368.0,上海,0
198,58.0,15039.0,深圳,0


In [28]:
# replace=False 不重复抽取，一定得到20个不同行
idx_age = np.random.choice(n, size=20, replace=False)
df.loc[idx_age, '年龄'] = np.nan

idx_income = np.random.choice(n, size=15, replace=False)
df.loc[idx_income, '收入'] = np.nan

In [29]:
df

,年龄,收入,城市,是否购买
0,56.0,47417.0,上海,0
1,46.0,26938.0,上海,0
2,32.0,26664.0,深圳,1
3,60.0,4636.0,深圳,0
4,25.0,23080.0,深圳,1
...,...,...,...,...
195,49.0,48091.0,深圳,1
196,49.0,7780.0,深圳,0
197,41.0,NaN,上海,0
198,58.0,NaN,深圳,0


In [30]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 200 entries, 0 to 199
Data columns (total 4 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   年龄      180 non-null    float64
 1   收入      185 non-null    float64
 2   城市      200 non-null    object 
 3   是否购买    200 non-null    int64  
dtypes: float64(2), int64(1), object(1)
memory usage: 6.4+ KB


In [31]:
X = df.drop('是否购买', axis=1)
y = df['是否购买']

In [32]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [33]:
X_train

,年龄,收入,城市
79,57.0,NaN,上海
197,41.0,NaN,上海
38,24.0,47357.0,上海
24,50.0,40157.0,上海
122,52.0,9893.0,深圳
...,...,...,...
106,52.0,26328.0,上海
14,41.0,49717.0,上海
92,62.0,5557.0,北京
179,18.0,3784.0,上海


In [34]:
X_test

,年龄,收入,城市
95,24.0,5200.0,上海
15,20.0,NaN,北京
30,NaN,37698.0,深圳
158,NaN,38743.0,北京
128,18.0,22738.0,深圳
115,25.0,47064.0,深圳
69,61.0,44976.0,深圳
170,61.0,10455.0,上海
174,28.0,38777.0,北京
45,31.0,44106.0,上海


In [35]:
len(y_train)

160

In [36]:
len(y_test)

40

In [37]:
numeric_features = ['年龄', '收入']
numeric_transformer = Pipeline(
    [
        ('imputer', SimpleImputer(strategy='median')),  # 中位数填充缺失值
        ('scaler', StandardScaler()),  # 标准化
    ]
)

In [38]:
categorical_features = ['城市']
categorical_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),  # 众数填充
    ('encoder', OneHotEncoder(handle_unknown='ignore')),  # 独热编码
])

In [39]:
preprocessor = ColumnTransformer(
    [
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features),
    ]
)

In [40]:
full_pipe = Pipeline(
    [
        ('preprocessor', preprocessor),
        ('classifier', KNeighborsClassifier(n_neighbors=5)),
    ]
)

In [41]:
full_pipe.fit(X_train, y_train)

,steps,"[('preprocessor', ...), ('classifier', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('num', ...), ('cat', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [42]:
print(full_pipe)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['年龄', '收入']),
                                                 ('cat',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('encoder',
                                                                   OneHotEncoder(handle_unknown='ignore'))]),
                                                  ['城市'])])),
      

In [43]:
y_pred = full_pipe.predict(X_test)

In [44]:
len(y_pred)

40

In [45]:
print(f"准确率: {accuracy_score(y_test, y_pred):.2%}")

准确率: 45.00%


In [46]:
print("\n分类报告:")
print(classification_report(y_test, y_pred))


分类报告:
              precision    recall  f1-score   support

           0       0.53      0.39      0.45        23
           1       0.39      0.53      0.45        17

    accuracy                           0.45        40
   macro avg       0.46      0.46      0.45        40
weighted avg       0.47      0.45      0.45        40

